## Действительно ли помогают неразмеченные данные?

Частичное обучение (semi-supervised learning) предлагает методы работы с выборками, в которых лишь для части объектов известны ответы. В статьях утверждается, что добавление неразмеченных данных позволяет повысить качество работы — давайте выясним, так ли это!

Наверное, проще всего добыть неразмеченные примеры, если речь идёт о работе с текстами или изображениями. Остановимся на текстах.

Будем работать с данными из соревнования Predict closed questions on Stack Overflow: https://www.kaggle.com/c/predict-closed-questions-on-stack-overflow/data

Нас будет интересовать файл train-sample.csv — загрузите его. Будем решать бинарную задачу: отнесём объект к классу 1, если `OpenStatus == 'open'`, и к классу 0 иначе.

**Задание 1. (5 баллов)**

Загрузите данные и подготовьте выборку. В качестве признаков возьмите TF-IDF по BodyMarkdown с `min_df=10`; про целевую переменную написано выше. Выделите тестовую выборку из 5000 объектов.

In [ ]:
#code here
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Загрузка данных
df = pd.read_csv('train-sample.csv')

# Целевая переменная: 1 если open, иначе 0
y = (df['OpenStatus'] == 'open').astype(int)

# Текстовые признаки
vectorizer = TfidfVectorizer(min_df=10, stop_words='english')
X = vectorizer.fit_transform(df['BodyMarkdown'])

# Разделение на обучающую+неразмеченную и тестовую выборки
X_train_pool, X_test, y_train_pool, y_test = train_test_split(
    X, y, test_size=5000, random_state=42, stratify=y
)

# Из обучающего пула выделим размеченную часть (например, 1000 объектов)
X_labeled, X_unlabeled, y_labeled, y_unlabeled = train_test_split(
    X_train_pool, y_train_pool, train_size=1000, random_state=42, stratify=y_train_pool
)

print(f"Размер размеченной выборки: {X_labeled.shape[0]}")
print(f"Размер неразмеченной выборки (из train_pool): {X_unlabeled.shape[0]}")
print(f"Размер тестовой выборки: {X_test.shape[0]}")

Нас будут интересовать качество (AUC-ROC) в четырёх следующих постановках:
1. Модель обучается только на размеченных данных.
2. Модель обучается на размеченных и неразмеченных данных, причём неразмеченная часть не пересекается с тестовой выборкой.
3. Модель обучается на размеченных и неразмеченных данных, причём неразмеченная часть совпадает с тестовой выборкой.
4. Модель обучается на размеченных и неразмеченных данных, причём неразмеченная часть включает в себя тестовую выборку.

**Задание 1. (5 баллов)**

Проведите эксперименты и сделайте выводы для любого из методов пакета `sklearn.semi_supervised` и для логистической регрессии

In [ ]:
#code here
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import LabelSpreading
from sklearn.metrics import roc_auc_score

# Словарь для хранения результатов
results = {}

# Базовая модель для логистической регрессии (одинаковые параметры во всех сценариях)
lr = LogisticRegression(max_iter=1000, random_state=42)

# --- Сценарий 1: только размеченные данные ---
lr.fit(X_labeled, y_labeled)
y_pred_prob = lr.predict_proba(X_test)[:, 1]
results['LR (only labeled)'] = roc_auc_score(y_test, y_pred_prob)

# --- Сценарий 2: размеченные + неразмеченные (не пересекающиеся с тестом) ---
# Объединяем размеченные и неразмеченные из train_pool, метки для неразмеченных = -1
X_train2 = np.vstack([X_labeled.toarray(), X_unlabeled.toarray()])
y_train2 = np.concatenate([y_labeled, [-1] * len(X_unlabeled)])

# Обучаем LabelSpreading
lp = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.8, max_iter=100)
lp.fit(X_train2, y_train2)
y_pred_prob = lp.predict_proba(X_test)[:, 1]
results['LabelSpreading (labeled + unlabeled)'] = roc_auc_score(y_test, y_pred_prob)

# --- Сценарий 3: размеченные + неразмеченные = тестовые данные ---
# Используем тестовые данные как неразмеченные
X_train3 = np.vstack([X_labeled.toarray(), X_test.toarray()])
y_train3 = np.concatenate([y_labeled, [-1] * len(X_test)])

lp = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.8, max_iter=100)
lp.fit(X_train3, y_train3)
y_pred_prob = lp.predict_proba(X_test)[:, 1]
results['LabelSpreading (labeled + test as unlabeled)'] = roc_auc_score(y_test, y_pred_prob)

# --- Сценарий 4: размеченные + неразмеченные, включающие тест ---
# Используем и unlabeled из train_pool, и тест
X_train4 = np.vstack([X_labeled.toarray(), X_unlabeled.toarray(), X_test.toarray()])
y_train4 = np.concatenate([y_labeled, [-1] * len(X_unlabeled), [-1] * len(X_test)])

lp = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.8, max_iter=100)
lp.fit(X_train4, y_train4)
y_pred_prob = lp.predict_proba(X_test)[:, 1]
results['LabelSpreading (labeled + unlabeled + test)'] = roc_auc_score(y_test, y_pred_prob)

# Вывод результатов
print("AUC-ROC на тестовой выборке:")
for name, score in results.items():
    print(f"{name}: {score:.4f}")

### Self-train

Обучаем на размеченной части, предсказываем неразмеченную, потом обучаем на всех, и предсказываем неразмеченную, повторяем пока не сойдёмся в предскзааниях неразмеченной части

In [ ]:
from sklearn.semi_supervised import SelfTrainingClassifier

# Self-training с базовой логистической регрессией
base_lr = LogisticRegression(max_iter=1000, random_state=42)
self_training = SelfTrainingClassifier(base_lr, threshold=0.75, criterion='threshold', max_iter=10)

# Обучаем на размеченных + неразмеченных из train_pool (сценарий 2, аналогично)
X_st = np.vstack([X_labeled.toarray(), X_unlabeled.toarray()])
y_st = np.concatenate([y_labeled, [-1] * len(X_unlabeled)])

self_training.fit(X_st, y_st)
y_pred_prob = self_training.predict_proba(X_test)[:, 1]
score_self = roc_auc_score(y_test, y_pred_prob)

print(f"Self-training (LR base) AUC-ROC: {score_self:.4f}")